# 【Day 06 實作】讓 Qwen 呼叫即時天氣工具

這份 Notebook 沿用 Day2 的 **Qwen2.5-0.5B-Instruct**，示範一個完整但精簡的 Function Calling 流程。

> Qwen 負責根據自然語言產生「工具請求」；真正連線 Open-Meteo 的，是下方的 Python 程式。

## Goal

執行完後，你會看見：

1. 使用者用自然語言問問題。
2. Qwen 判斷是否需要 `get_weather`。
3. Python 驗證工具名稱與城市參數。
4. Python 查詢 Open-Meteo 的即時資料。
5. Qwen 根據工具結果整理繁體中文回答。

本範例使用免 API Key 的 [Open-Meteo](https://open-meteo.com/)，但執行時仍需要網路。

## Setup

第一次執行時，套件安裝與模型載入可能需要幾分鐘。Qwen 模型約 1 GB，不需要 Hugging Face Token。

In [1]:
%pip install -q torch transformers accelerate requests nbformat

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import re
from typing import Any

import requests
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [3]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
model.eval()

print(f"Model: {model_name}")
print(f"Device: {device}")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model: Qwen/Qwen2.5-0.5B-Instruct
Device: mps


## Define the Tool

`get_weather(city)` 會先把城市名稱轉成座標，再查詢目前天氣。  
特別處理「臺北」是因為 Open-Meteo 的地名搜尋能找到「台北市」，卻不一定能直接找到「臺北」。

工具回傳的是資料，不是自然語言答案。

In [4]:
import json
import re
from typing import Any

import requests


ALLOWED_ACTIONS = {"get_weather", "answer"}
GEOCODING_URL = "https://geocoding-api.open-meteo.com/v1/search"
FORECAST_URL = "https://api.open-meteo.com/v1/forecast"
CURRENT_VARIABLES = (
    "temperature_2m",
    "apparent_temperature",
    "relative_humidity_2m",
    "precipitation",
    "weather_code",
    "wind_speed_10m",
)

WEATHER_CODE_LABELS = {
    0: "晴朗",
    1: "大致晴朗",
    2: "局部多雲",
    3: "陰天",
    45: "有霧",
    48: "霧淞",
    51: "小毛毛雨",
    53: "中度毛毛雨",
    55: "大毛毛雨",
    56: "輕微凍毛毛雨",
    57: "強凍毛毛雨",
    61: "小雨",
    63: "中雨",
    65: "大雨",
    66: "輕微凍雨",
    67: "強凍雨",
    71: "小雪",
    73: "中雪",
    75: "大雪",
    77: "霰",
    80: "小陣雨",
    81: "中度陣雨",
    82: "強陣雨",
    85: "小陣雪",
    86: "大陣雪",
    95: "雷雨",
    96: "雷雨伴隨小冰雹",
    99: "雷雨伴隨大冰雹",
}

CITY_QUERY_ALIASES = {
    "臺北": "台北市",
    "台北": "台北市",
    "臺北市": "台北市",
}


def normalize_city_query(city: str) -> str:
    normalized = city.strip()
    return CITY_QUERY_ALIASES.get(normalized, normalized)


def extract_json_object(text: str) -> dict[str, Any]:
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip())
    start = cleaned.find("{")
    end = cleaned.rfind("}")
    if start == -1 or end == -1 or end < start:
        raise ValueError("模型沒有回傳 JSON 物件")
    parsed = json.loads(cleaned[start : end + 1])
    if not isinstance(parsed, dict):
        raise ValueError("工具請求必須是 JSON 物件")
    return parsed


def parse_tool_request(text: str) -> dict[str, Any]:
    request_data = extract_json_object(text)
    action = request_data.get("action")
    if action not in ALLOWED_ACTIONS:
        raise ValueError(f"不允許的 action：{action}")

    if action == "get_weather":
        city = request_data.get("arguments", {}).get("city")
        if not isinstance(city, str) or not city.strip():
            raise ValueError("get_weather 需要非空白的 city")
        return {"action": action, "arguments": {"city": city.strip()}}

    answer = request_data.get("answer")
    if not isinstance(answer, str) or not answer.strip():
        raise ValueError("answer action 需要文字答案")
    return {"action": action, "answer": answer.strip()}


def _get_json(url: str, params: dict[str, Any]) -> dict[str, Any]:
    try:
        response = requests.get(url, params=params, timeout=15)
        response.raise_for_status()
        payload = response.json()
    except requests.Timeout as error:
        raise RuntimeError("天氣服務回應逾時，請稍後再試") from error
    except requests.RequestException as error:
        raise RuntimeError(f"無法連接天氣服務：{error}") from error
    except ValueError as error:
        raise RuntimeError("天氣服務沒有回傳合法 JSON") from error
    if not isinstance(payload, dict):
        raise RuntimeError("天氣服務回傳格式不正確")
    return payload


def get_weather(city: str) -> dict[str, Any]:
    if not isinstance(city, str) or not city.strip():
        raise ValueError("city 不可為空白")

    geocoding_data = _get_json(
        GEOCODING_URL,
        {
            "name": normalize_city_query(city),
            "count": 1,
            "language": "zh",
            "format": "json",
        },
    )
    locations = geocoding_data.get("results") or []
    if not locations:
        raise LookupError(f"找不到城市：{city}")

    location = locations[0]
    latitude = location.get("latitude")
    longitude = location.get("longitude")
    if latitude is None or longitude is None:
        raise RuntimeError("地理編碼結果缺少經緯度")

    forecast_data = _get_json(
        FORECAST_URL,
        {
            "latitude": latitude,
            "longitude": longitude,
            "current": ",".join(CURRENT_VARIABLES),
            "timezone": "auto",
        },
    )
    current = forecast_data.get("current")
    if not isinstance(current, dict):
        raise RuntimeError("天氣服務回傳結果缺少 current 欄位")

    weather_code = current.get("weather_code")
    current_result = {
        **current,
        "weather_description": WEATHER_CODE_LABELS.get(
            weather_code,
            f"未知天氣代碼 {weather_code}",
        ),
    }

    return {
        "city": location.get("name", city.strip()),
        "country": location.get("country"),
        "latitude": latitude,
        "longitude": longitude,
        "timezone": forecast_data.get("timezone"),
        "observed_at": current.get("time"),
        "current": current_result,
        "units": forecast_data.get("current_units", {}),
        "source": "Open-Meteo",
        "source_urls": {
            "geocoding": GEOCODING_URL,
            "forecast": FORECAST_URL,
        },
    }


def _measurement(
    weather_result: dict[str, Any],
    field: str,
) -> str:
    value = weather_result["current"][field]
    unit = weather_result.get("units", {}).get(field, "")
    return f"{value}{unit}"


def validate_weather_answer(
    answer: str,
    weather_result: dict[str, Any],
) -> list[str]:
    required_facts = {
        "城市": str(weather_result["city"]),
        "觀測時間": str(weather_result["observed_at"]),
        "天氣描述": str(
            weather_result["current"]["weather_description"]
        ),
        "溫度": _measurement(weather_result, "temperature_2m"),
        "體感溫度": _measurement(
            weather_result,
            "apparent_temperature",
        ),
        "濕度": _measurement(
            weather_result,
            "relative_humidity_2m",
        ),
        "目前降水量": _measurement(
            weather_result,
            "precipitation",
        ),
        "風速": _measurement(weather_result, "wind_speed_10m"),
    }
    errors = []
    for label, expected_text in required_facts.items():
        if expected_text not in answer:
            if label == "天氣描述":
                errors.append(
                    f"未忠實保留{label}「{expected_text}」"
                )
            else:
                errors.append(f"未忠實保留{label} {expected_text}")
    return errors


def build_verified_weather_answer(
    weather_result: dict[str, Any],
) -> str:
    current = weather_result["current"]
    apparent_temperature = current["apparent_temperature"]
    if apparent_temperature >= 30:
        clothing_advice = (
            "體感偏熱，一般不需要外套，請注意補充水分。"
        )
    elif apparent_temperature <= 18:
        clothing_advice = "體感偏涼，建議攜帶外套。"
    else:
        clothing_advice = "是否攜帶薄外套可依個人感受決定。"

    return (
        f"{weather_result['city']}在"
        f"{weather_result['observed_at']}的天氣為"
        f"{current['weather_description']}，"
        f"溫度{_measurement(weather_result, 'temperature_2m')}，"
        f"體感溫度"
        f"{_measurement(weather_result, 'apparent_temperature')}，"
        f"濕度"
        f"{_measurement(weather_result, 'relative_humidity_2m')}，"
        f"目前降水量"
        f"{_measurement(weather_result, 'precipitation')}，"
        f"風速{_measurement(weather_result, 'wind_speed_10m')}。"
        f"{clothing_advice}"
        "這項資料只代表目前降水量，不代表未來是否下雨。"
    )


### 先檢查解析與安全邊界

模型輸出不能直接執行。這裡只允許 `get_weather` 與 `answer`，未知 action 會被拒絕。

In [5]:
assert parse_tool_request(
    '{"action":"get_weather","arguments":{"city":"臺北"}}'
) == {"action": "get_weather", "arguments": {"city": "臺北"}}

assert parse_tool_request(
    '```json\n{"action":"answer","answer":"你好"}\n```'
) == {"action": "answer", "answer": "你好"}

try:
    parse_tool_request('{"action":"delete_file","arguments":{"path":"/"}}')
except ValueError as error:
    assert "不允許的 action" in str(error)
else:
    raise AssertionError("未知 action 應被拒絕")

assert normalize_city_query("臺北") == "台北市"
print("解析與白名單檢查：通過")

解析與白名單檢查：通過


## Ask Qwen to Choose

我們把「有哪些工具」與允許的 JSON 格式放進 System Prompt。  
Qwen 只能提出請求，這一格還不會查天氣。

In [6]:
def generate_text(
    messages: list[dict[str, str]],
    max_new_tokens: int = 180,
) -> str:
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    model_inputs = tokenizer([prompt], return_tensors="pt").to(device)

    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_token_ids = generated_ids[:, model_inputs.input_ids.shape[1] :]
    return tokenizer.batch_decode(
        new_token_ids,
        skip_special_tokens=True,
    )[0].strip()


TOOL_ROUTER_PROMPT = """
你是工具路由器，只能輸出一個 JSON 物件，不可輸出 Markdown 或解釋。

唯一可用工具：
- get_weather(city)：查詢指定城市目前的即時天氣。

規則：
1. 問題需要目前或即時天氣資料時，輸出：
   {"action":"get_weather","arguments":{"city":"城市名稱"}}
2. 問題不需要即時天氣時，直接簡短回答並輸出：
   {"action":"answer","answer":"繁體中文回答"}
3. 不可捏造即時天氣，不可使用其他 action。

範例：
使用者：高雄現在幾度？
輸出：{"action":"get_weather","arguments":{"city":"高雄市"}}

使用者：什麼是 Function Calling？
輸出：{"action":"answer","answer":"Function Calling 是讓模型提出工具名稱與參數，再由外部程式實際執行的方法。"}
""".strip()


def generate_tool_request(
    question: str,
) -> tuple[str, dict[str, Any]]:
    messages = [
        {"role": "system", "content": TOOL_ROUTER_PROMPT},
        {"role": "user", "content": question},
    ]
    raw_text = generate_text(messages, max_new_tokens=120)

    try:
        return raw_text, parse_tool_request(raw_text)
    except (ValueError, json.JSONDecodeError) as first_error:
        repair_messages = [
            {
                "role": "system",
                "content": (
                    "你只負責修正格式。請把內容修正成一個合法 JSON 物件，"
                    "action 只能是 get_weather 或 answer，不可加入解釋。"
                ),
            },
            {
                "role": "user",
                "content": f"原始問題：{question}\n待修正內容：{raw_text}",
            },
        ]
        repaired_text = generate_text(repair_messages, max_new_tokens=120)
        try:
            return repaired_text, parse_tool_request(repaired_text)
        except (ValueError, json.JSONDecodeError) as second_error:
            raise ValueError(
                "Qwen 兩次都沒有產生合法工具請求。"
                f"\n第一次輸出：{raw_text}"
                f"\n第二次輸出：{repaired_text}"
            ) from second_error

## Run the Tool

先單獨測試真正的 API。這一格會連線兩次：

1. Geocoding API：把「臺北」轉成經緯度。
2. Forecast API：用經緯度取得目前天氣。

輸出保留來源與觀測時間，方便確認 AI 不是「憑感覺報天氣」。

In [7]:
weather_smoke_test = get_weather("臺北")
assert weather_smoke_test["city"]
assert "temperature_2m" in weather_smoke_test["current"]
assert weather_smoke_test["source"] == "Open-Meteo"

print(json.dumps(weather_smoke_test, ensure_ascii=False, indent=2))

{
  "city": "台北市",
  "country": "台湾",
  "latitude": 25.05306,
  "longitude": 121.52639,
  "timezone": "Asia/Taipei",
  "observed_at": "2026-08-03T13:45",
  "current": {
    "time": "2026-08-03T13:45",
    "interval": 900,
    "temperature_2m": 35.8,
    "apparent_temperature": 39.5,
    "relative_humidity_2m": 42,
    "precipitation": 0.0,
    "weather_code": 3,
    "wind_speed_10m": 15.5,
    "weather_description": "陰天"
  },
  "units": {
    "time": "iso8601",
    "interval": "seconds",
    "temperature_2m": "°C",
    "apparent_temperature": "°C",
    "relative_humidity_2m": "%",
    "precipitation": "mm",
    "weather_code": "wmo code",
    "wind_speed_10m": "km/h"
  },
  "source": "Open-Meteo",
  "source_urls": {
    "geocoding": "https://geocoding-api.open-meteo.com/v1/search",
    "forecast": "https://api.open-meteo.com/v1/forecast"
  }
}


## Return the Result to Qwen

工具完成後，再把有來源的資料交回模型。這次 Qwen 的工作是整理，不是猜測。

不過，小模型即使收到正確資料，仍可能抄錯數字或自行補充天氣。因此這裡會保留 Qwen 草稿，
再由 Python 檢查它有沒有忠實保留關鍵事實；最後顯示的答案則由已驗證的工具值組合而成。

In [8]:
def generate_final_answer(
    question: str,
    weather_result: dict[str, Any],
) -> str:
    messages = [
        {
            "role": "system",
            "content": (
                "你是天氣助理。只根據工具結果，以繁體中文簡潔回答。"
                "說明城市、觀測時間、天氣、溫度、體感溫度、濕度與風速。"
                "可以依溫度提供保守的穿著建議，但不可捏造降雨機率；"
                "工具只有 precipitation，不代表未來降雨機率。"
            ),
        },
        {
            "role": "user",
            "content": (
                f"使用者問題：{question}\n"
                "Open-Meteo 工具結果：\n"
                f"{json.dumps(weather_result, ensure_ascii=False)}"
            ),
        },
    ]
    return generate_text(messages, max_new_tokens=220)


def run_agent(user_question: str) -> dict[str, Any]:
    raw_request, tool_request = generate_tool_request(user_question)
    trace = {
        "question": user_question,
        "raw_model_output": raw_request,
        "tool_request": tool_request,
    }

    if tool_request["action"] == "answer":
        trace["final_answer"] = tool_request["answer"]
        return trace

    weather_result = get_weather(tool_request["arguments"]["city"])
    trace["tool_result"] = weather_result
    model_draft = generate_final_answer(
        user_question,
        weather_result,
    )
    validation_errors = validate_weather_answer(
        model_draft,
        weather_result,
    )
    trace["model_draft"] = model_draft
    trace["answer_validation"] = {
        "passed": not validation_errors,
        "issues": validation_errors,
    }
    trace["final_answer"] = build_verified_weather_answer(
        weather_result,
    )
    return trace

## Checks

### 案例一：需要即時工具

完整 trace 會顯示模型原始輸出、通過驗證的工具請求、API 結果與最後回答。

In [9]:
weather_demo = run_agent("臺北現在天氣如何？出門需要帶外套嗎？")
print(json.dumps(weather_demo, ensure_ascii=False, indent=2))

if weather_demo["tool_request"]["action"] != "get_weather":
    print("注意：這次小模型沒有選擇天氣工具，請重新執行本格觀察差異。")

{
  "question": "臺北現在天氣如何？出門需要帶外套嗎？",
  "raw_model_output": "{\"action\":\"get_weather\",\"arguments\":{\"city\":\"台北市\"}}",
  "tool_request": {
    "action": "get_weather",
    "arguments": {
      "city": "台北市"
    }
  },
  "tool_result": {
    "city": "台北市",
    "country": "台湾",
    "latitude": 25.05306,
    "longitude": 121.52639,
    "timezone": "Asia/Taipei",
    "observed_at": "2026-08-03T13:45",
    "current": {
      "time": "2026-08-03T13:45",
      "interval": 900,
      "temperature_2m": 35.8,
      "apparent_temperature": 39.5,
      "relative_humidity_2m": 42,
      "precipitation": 0.0,
      "weather_code": 3,
      "wind_speed_10m": 15.5,
      "weather_description": "陰天"
    },
    "units": {
      "time": "iso8601",
      "interval": "seconds",
      "temperature_2m": "°C",
      "apparent_temperature": "°C",
      "relative_humidity_2m": "%",
      "precipitation": "mm",
      "weather_code": "wmo code",
      "wind_speed_10m": "km/h"
    },
    "source": "Open-Mete

### 案例二：不需要工具

這個問題不需要即時外部資料，理想情況下 Qwen 應直接回答。

In [9]:
direct_demo = run_agent("請用一句話解釋什麼是 Function Calling。")
print(json.dumps(direct_demo, ensure_ascii=False, indent=2))

if direct_demo["tool_request"]["action"] != "answer":
    print("注意：這次小模型誤選了天氣工具，這也是小模型工具路由的限制。")

{
  "question": "請用一句話解釋什麼是 Function Calling。",
  "raw_model_output": "{\"action\":\"answer\",\"answer\":\"Function Calling 是讓模型提出工具名稱與參數，再由外部程式實際執行的方法。\"}",
  "tool_request": {
    "action": "answer",
    "answer": "Function Calling 是讓模型提出工具名稱與參數，再由外部程式實際執行的方法。"
  },
  "final_answer": "Function Calling 是讓模型提出工具名稱與參數，再由外部程式實際執行的方法。"
}


## Next Steps

這個實驗刻意把每一層攤開，因此可以觀察到：

- **使用者指令會影響選擇**：問即時天氣時，模型應選 `get_weather`；概念題則直接回答。
- **Function Call 只是請求**：Qwen 輸出的 JSON 不會自己連上網路，Python 才真正執行 API。
- **工具需要白名單與驗證**：未知 action、空白城市或錯誤格式都會被拒絕。
- **小模型不是每次都穩定**：0.5B 模型可能選錯工具或輸出不合法 JSON，所以範例加入一次格式修正機會。
- **拿到工具結果也不等於一定會抄對**：Notebook 保留 Qwen 草稿與驗證結果，最後答案使用經 Python 核對的工具值。
- **高風險工具要更保守**：若工具改成寄信、刪檔或付款，執行前應加入權限檢查與人類確認。

可以嘗試更換城市、修改問題，或刻意要求「不要查網路」，觀察自然語言如何影響模型的工具選擇。

### 資料來源

- [Open-Meteo Geocoding API](https://open-meteo.com/en/docs/geocoding-api)
- [Open-Meteo Weather Forecast API](https://open-meteo.com/en/docs)